In [1]:
!pip install optuna
!pip install torch

In [2]:
import os
import numpy as np
import pandas as pd
import boto3
import time
import helper
import sys
import matplotlib.pyplot as plt
cwd = os.getcwd()
print(cwd)

from sagemaker import get_execution_role

from Pumpkin.metrics import KeyMetrics # internal tool for risk model evaluation 

import optuna
from optuna.samplers import GPSampler, TPESampler

/home/sagemaker-user/CAPE_PERFORMANCE


/opt/conda/envs/model-340/lib/python3.12/site-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/envs/model-340/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.linear_model import TweedieRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

#### Load train, val, and holdout (test) dataset

We divide the train into new_train (0.5), new_valid (0.2), and make the valid labels become our holdout (0.3).

In [4]:
version = 'v5'

SPLIT_TRAIN = 1
# SPLIT_TRAIN = 0 is to split valid (0.3) into valid (0.2) and holdout (0.1), 
# the samples are not enought for statistical analysis, so we don't use it eventually.


if SPLIT_TRAIN == 1:
    train = pd.read_csv(f'./{version}_datasets/train.csv')
    test = pd.read_csv(f'./{version}_datasets/val.csv')
    val = pd.read_csv(f'./{version}_datasets/test.csv')
elif SPLIT_TRAIN ==0:
    train = pd.read_csv(f'./{version}_datasets_val_split/train.csv')
    val = pd.read_csv(f'./{version}_datasets_val_split/val.csv')
    test = pd.read_csv(f'./{version}_datasets_val_split/test.csv')

/tmp/ipykernel_11940/135506397.py:7: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  test = pd.read_csv(f'./{version}_datasets/val.csv')


#### Remove the unnecessary variables

In [5]:
helper.keywords_in_var('adj', train)
print(len(train.columns))
print((train.columns))

keywords are not in iterable
There are  2  variables contains adj.
ncat_infl_adj 
pp_infl_adj
136
Index(['Unnamed: 0', 'cape_response_status', 'cape_response_description',
       'cape_response_id', 'cape_primary_structure_latitude',
       'cape_primary_structure_longitude', 'attribute_geometry_id',
       'cape_parcel_id', 'cape_oblique_property_date',
       'cape_oblique_property_image_source',
       ...
       'new_or_rnwl_flg', 'zip', 'zip4', 'year', 'tv', 'form',
       'cumulative_inflation_factor', 'ncat_infl_adj', 'pp_infl_adj',
       'p0_alxc_wo_cape_pred'],
      dtype='object', length=136)


Remove the time, address, id, source, url, zip related variables.

In [6]:
EXCLUDE_EXACT = ['Unnamed: 0',"qpid","pol_num","cur_term_eff_dt","rundt", 'property_address',
                'city', 'county', 'orgl_pol_eff_dt', 'cape_geocode_method',
                'cape_run_dt', 'address1','source', 'source row number',
                'cape_response_status', 'cape_response_description', 
                # 'cape_primary_structure_latitude','cape_primary_structure_longitude',
                # 'cape_yard_debris_coverage_sqft', 'cape_accessory_structure_footprint',
                # 'cape_roof_condition_reasons',
                'state', 'year', 
                'co_cd',]
EXCLUDE_SUB   = ['_date', '_url', '_id', 'zip', '_source',
                # '_confidence',
                'yard', 'pool', 'loose',
                ]   # “*ncat*” as substring


def select_features(df: pd.DataFrame) -> pd.DataFrame:
    keep = []
    for c in df.columns:
        c_lower = c.lower()  # lower-case column name
        # Check exact and substring matches in lower-case
        if c_lower in EXCLUDE_EXACT:
            continue
        if any(sub in c_lower for sub in EXCLUDE_SUB):
            continue
        keep.append(c)

    # Keep only numeric columns, coerce safely
    X = df[keep]
    return X

In [7]:
train_select = select_features(train).copy()
val_select   = select_features(val).reindex(columns=train_select.columns).copy()
test_select = select_features(test).reindex(columns=train_select.columns).copy()

train_select = train_select.drop(columns=['Unnamed: 0'], errors='ignore')
val_select = val_select.drop(columns=['Unnamed: 0'], errors='ignore')
test_select = test_select.drop(columns=['Unnamed: 0'], errors='ignore')


Check the results

In [8]:
print(len(train_select), len(train_select)/(len(train_select)+len(val_select)+len(test_select)))
print(len(val_select), len(val_select)/(len(train_select)+len(val_select)+len(test_select)))
print(len(test_select), len(test_select)/(len(train_select)+len(val_select)+len(test_select)))

409353 0.4909816442457157
175438 0.21042190408566658
248953 0.2985964516686177


In [9]:
# print(train_select['cape_roof_condition_reasons'].unique())

In [10]:
print(train_select.columns)
print(len(train_select.columns))

Index(['cape_primary_structure_latitude', 'cape_primary_structure_longitude',
       'cape_geocode_confidence', 'cape_accessory_structure_count',
       'cape_accessory_structure_footprint',
       'cape_accessory_structure_roof_condition_rating',
       'cape_accessory_structure_roof_condition_rating_confidence',
       'cape_roof_condition_rating', 'cape_roof_condition_rating_confidence',
       'cape_roof_condition_reasons', 'cape_roof_effluent_runoff',
       'cape_roof_effluent_runoff_confidence',
       'cape_roof_material_degradation',
       'cape_roof_material_degradation_confidence',
       'cape_roof_material_degradation_reason',
       'cape_roof_missing_or_peeling_material',
       'cape_roof_missing_or_peeling_material_confidence',
       'cape_roof_natural_discoloration',
       'cape_roof_natural_discoloration_confidence', 'cape_roof_patching',
       'cape_roof_patching_confidence', 'cape_roof_ponding',
       'cape_roof_ponding_confidence', 'cape_roof_rusting',
      

#### Baseline Gini score performance

In [11]:
pred_m140_t = train_select['p0_alxc_wo_cape_pred']
pred_m140_v = val_select['p0_alxc_wo_cape_pred']
ee_t = train_select['ee']
ee_v = val_select['ee']

km_t_base = KeyMetrics(train_select['ncat_infl_adj']/ee_t, # actual pp
                   pred_m140_t, # predicted pp
                   ee_t)

# initiate KeyMetrics for this validation set:
km_v_base = KeyMetrics(val_select['ncat_infl_adj']/ee_v, # actual pp
                   pred_m140_v, # predicted pp
                   ee_v)
km_t_base.gini(), km_v_base.gini() 

(0.3815020376908176, 0.450875115181504)

#### Model

In [12]:
### The following code is used to obtain the categorical columns,
### Only need to be run for once.
def inspect_categoricals(df, *, low_card_ratio=0.02, low_card_max=50, include_bool=True):
    n = len(df)
    obj_cols  = df.select_dtypes(include=["object"]).columns.tolist()
    cat_cols  = df.select_dtypes(include=["category"]).columns.tolist()
    bool_cols = df.select_dtypes(include=["bool"]).columns.tolist() if include_bool else []
    num_cols  = df.select_dtypes(include=["number"]).columns

    obvious = obj_cols + cat_cols + bool_cols

    nunq = df[num_cols].nunique(dropna=False)
    low_card_num = nunq[(nunq <= low_card_max) | (nunq / max(n,1) <= low_card_ratio)].index.tolist()

    print("Obvious categorical:", obvious)
    print("Numeric low-cardinality candidates:", low_card_num)

    # small table
    summ = (
        df[obvious + low_card_num]
        .nunique(dropna=False)
        .rename("unique")
        .to_frame()
        .assign(pct_unique=lambda s: s["unique"] / max(n, 1))
        .assign(dtype=lambda s: [str(df[c].dtype) for c in s.index])
        .sort_values(["dtype", "unique"])
    )
    return {"obvious": obvious, "num_low_card": low_card_num, "summary": summ}

res = inspect_categoricals(train_select)
res["summary"].head(20)

Obvious categorical: ['cape_geocode_confidence', 'cape_accessory_structure_roof_condition_rating', 'cape_roof_condition_rating', 'cape_roof_condition_reasons', 'cape_roof_effluent_runoff', 'cape_roof_material_degradation', 'cape_roof_material_degradation_reason', 'cape_roof_missing_or_peeling_material', 'cape_roof_natural_discoloration', 'cape_roof_patching', 'cape_roof_ponding', 'cape_roof_rusting', 'cape_roof_sealing', 'cape_roof_streaking', 'cape_roof_structural_damage', 'cape_roof_tarp', 'tv', 'form']
Numeric low-cardinality candidates: ['cape_accessory_structure_count', 'cape_accessory_structure_footprint', 'cape_accessory_structure_roof_condition_rating_confidence', 'cape_roof_condition_rating_confidence', 'cape_roof_effluent_runoff_confidence', 'cape_roof_material_degradation_confidence', 'cape_roof_missing_or_peeling_material_confidence', 'cape_roof_natural_discoloration_confidence', 'cape_roof_patching_confidence', 'cape_roof_ponding_confidence', 'cape_roof_rusting_confidence'

,unique,pct_unique,dtype
new_or_rnwl_flg,2,0.000005,float64
ncat_cnt,5,0.000012,float64
cumulative_inflation_factor,10,0.000024,float64
cape_accessory_structure_count,20,0.000049,float64
cape_accessory_structure_footprint,575,0.001405,float64
ee,4217,0.010302,float64
cape_roof_structural_damage_confidence,4429,0.010820,float64
cape_roof_rusting_confidence,4525,0.011054,float64
cape_roof_tarp_confidence,4630,0.011311,float64
cape_roof_ponding_confidence,5098,0.012454,float64


In [13]:
if version == 'v4': 
    cols_to_cat = [
        'cape_geocode_confidence', 'cape_accessory_structure_roof_condition_rating', 
        'cape_roof_condition_rating', 'cape_roof_condition_reasons', 
        'cape_roof_material_degradation', 'cape_roof_material_degradation_reason', 
        'cape_roof_natural_discoloration', 'cape_roof_patching', 
        'cape_roof_ponding', 'cape_roof_streaking', 'cape_roof_tarp'
    ]

if version == 'v5':
    cols_to_cat = [
    'cape_geocode_confidence', 'cape_accessory_structure_roof_condition_rating', 
    'cape_roof_condition_rating', 'cape_roof_effluent_runoff',
     'cape_roof_material_degradation', 'cape_roof_material_degradation_reason', 
     'cape_roof_missing_or_peeling_material', 'cape_roof_natural_discoloration',
      'cape_roof_patching', 'cape_roof_ponding', 'cape_roof_rusting', 'cape_roof_sealing', 
    'cape_roof_streaking', 'cape_roof_structural_damage', 'cape_roof_tarp',
    # 'cape_roof_condition_reasons',
    ]

for cat_col in cols_to_cat:
    print(f'For variable {cat_col}, there are unique results of {train_select[cat_col].unique()}')

For variable cape_geocode_confidence, there are unique results of ['optimal_target_match' 'good_other_structure_match' 'good_target_match'
 'uncertain' 'parcel_match']
For variable cape_accessory_structure_roof_condition_rating, there are unique results of [nan '0' '1' '1.0' '2' '2.0' '-1' '0.0' 'unknown' '-1.0' '-2' '-2.0']
For variable cape_roof_condition_rating, there are unique results of ['2' '-1' '1' 'unknown' '0' '-2' nan]
For variable cape_roof_effluent_runoff, there are unique results of ['no_roof_effluent_runoff' 'with_roof_effluent_runoff' 'unknown' nan]
For variable cape_roof_material_degradation, there are unique results of ['no_roof_material_degradation' 'roof_material_degradation_minor'
 'unknown' 'roof_material_degradation_major' nan]
For variable cape_roof_material_degradation_reason, there are unique results of [nan 'roof_effluent_runoff' 'roof_missing_or_peeling_material'
 'roof_sealing' 'roof_rusting'
 'roof_missing_or_peeling_material,roof_effluent_runoff'
 'roof_r

In [14]:
col = "cape_accessory_structure_roof_condition_rating"
mapping_str = {'0.0': '0', '1.0': '-1', '2.0': '1', '-1.0': '-2', '-2.0': '2'}
train_select[col] = train_select[col].replace(mapping_str)
val_select[col] = val_select[col].replace(mapping_str)
test_select[col] = test_select[col].replace(mapping_str)
print(f'For variable {col}, there are unique results of {train_select[col].unique()}')

For variable cape_accessory_structure_roof_condition_rating, there are unique results of [nan '0' '1' '-1' '2' 'unknown' '-2']


In [15]:
# mark them as cateory for XGBoost
for cat_col in cols_to_cat:
    train_select[cat_col] = train_select[cat_col].astype('category')
    val_select[cat_col] = val_select[cat_col].astype('category')
    test_select[cat_col] = test_select[cat_col].astype('category')

In [16]:
if version == 'v4':
       ls_predictors = [
       'cape_primary_structure_latitude', 'cape_primary_structure_longitude',
       'cape_geocode_confidence', 'cape_accessory_structure_count',
       'cape_accessory_structure_footprint',
       'cape_accessory_structure_roof_condition_rating',
       'cape_accessory_structure_roof_condition_rating_confidence',
       'cape_roof_condition_rating', 'cape_roof_condition_rating_confidence',
       'cape_roof_condition_reasons', 'cape_roof_material_degradation',
       'cape_roof_material_degradation_confidence',
       'cape_roof_material_degradation_reason',
       'cape_roof_material_degradation_reason_confidence',
       'cape_roof_natural_discoloration',
       'cape_roof_natural_discoloration_confidence', 'cape_roof_patching',
       'cape_roof_patching_confidence', 'cape_roof_ponding',
       'cape_roof_ponding_confidence', 'cape_roof_streaking',
       'cape_roof_streaking_confidence', 'cape_roof_tarp',
       'cape_roof_tarp_confidence'
       ]
if version == 'v5':
       # ls_predictors = [
       #        'cape_geocode_confidence', 'cape_accessory_structure_count',
       #        'cape_accessory_structure_footprint',
       #        'cape_accessory_structure_roof_condition_rating',
       #        'cape_accessory_structure_roof_condition_rating_confidence',
       #        'cape_roof_condition_rating', 'cape_roof_condition_rating_confidence',
       #        'cape_roof_condition_reasons', 'cape_roof_effluent_runoff',
       #        'cape_roof_effluent_runoff_confidence',
       #        'cape_roof_material_degradation',
       #        'cape_roof_material_degradation_confidence',
       #        'cape_roof_material_degradation_reason',
       #        'cape_roof_missing_or_peeling_material',
       #        'cape_roof_missing_or_peeling_material_confidence',
       #        'cape_roof_natural_discoloration',
       #        'cape_roof_natural_discoloration_confidence', 'cape_roof_patching',
       #        'cape_roof_patching_confidence', 'cape_roof_ponding',
       #        'cape_roof_ponding_confidence', 'cape_roof_rusting',
       #        'cape_roof_rusting_confidence', 'cape_roof_sealing',
       #        'cape_roof_sealing_confidence', 'cape_roof_streaking',
       #        'cape_roof_streaking_confidence', 'cape_roof_structural_damage',
       #        'cape_roof_structural_damage_confidence', 'cape_roof_tarp',
       #        'cape_roof_tarp_confidence',]


       ls_predictors = [             
              # 'cape_roof_material_degradation_reason',
              # 'cape_roof_condition_reasons',
              'cape_primary_structure_latitude', 'cape_primary_structure_longitude',
              'cape_geocode_confidence', 'cape_accessory_structure_count',
              'cape_accessory_structure_footprint',
              'cape_accessory_structure_roof_condition_rating',
              'cape_accessory_structure_roof_condition_rating_confidence',
              'cape_roof_condition_rating', 'cape_roof_condition_rating_confidence',
              'cape_roof_effluent_runoff', 'cape_roof_effluent_runoff_confidence',
              'cape_roof_material_degradation',
              'cape_roof_material_degradation_confidence',
              'cape_roof_missing_or_peeling_material',
              'cape_roof_missing_or_peeling_material_confidence',
              'cape_roof_natural_discoloration',
              'cape_roof_natural_discoloration_confidence', 'cape_roof_patching',
              'cape_roof_patching_confidence', 'cape_roof_ponding',
              'cape_roof_ponding_confidence', 'cape_roof_rusting',
              'cape_roof_rusting_confidence', 'cape_roof_sealing',
              'cape_roof_sealing_confidence', 'cape_roof_streaking',
              'cape_roof_streaking_confidence', 'cape_roof_structural_damage',
              'cape_roof_structural_damage_confidence', 'cape_roof_tarp',
              'cape_roof_tarp_confidence'
       ]
       
print('Final variables length', len(ls_predictors))

Final variables length 31


In [17]:
X_t = train_select[ ls_predictors]
y_t = train_select[ 'ncat_infl_adj']
ee_t = train_select[ 'ee']

X_v = val_select[ ls_predictors]
y_v = val_select[ 'ncat_infl_adj']
ee_v = val_select[ 'ee']

X_test = test_select[ ls_predictors]
y_test = test_select[ 'ncat_infl_adj']
ee_test = test_select[ 'ee']


We don't want infinite in modeling dataset, check this before training the model.

In [18]:
# 1) Fast boolean check (numeric columns only)
has_inf = np.isinf(X_v.select_dtypes(include=["number"]).to_numpy()).any()
print("Any ±inf in numeric columns?", bool(has_inf))

# 2) Per-column counts (numeric columns)
num = X_v.select_dtypes(include=["number"])
inf_counts = pd.DataFrame({
    "pos_inf": np.isposinf(num).sum(),
    "neg_inf": np.isneginf(num).sum(),
    "any_inf":  np.isinf(num).sum(),
})
print(inf_counts[inf_counts.any(axis=1)])

# 3) Row indices with any ±inf (numeric columns)
rows_with_inf = num.index[np.isinf(num.to_numpy()).any(axis=1)]
print("Rows with ±inf (first 10):", list(rows_with_inf[:10]))

Any ±inf in numeric columns? False
Empty DataFrame
Columns: [pos_inf, neg_inf, any_inf]
Index: []
Rows with ±inf (first 10): []


In [19]:
D_t = xgb.DMatrix(X_t, label=y_t, weight=ee_t, enable_categorical=True) 
D_t.set_base_margin(np.log(ee_t*pred_m140_t)) # offset approach of risk modeling, some papers suggest better than directly model pure premium (loss/ee) as target

D_v = xgb.DMatrix(X_v, label=y_v, weight=ee_v, enable_categorical=True) 
D_v.set_base_margin(np.log(ee_v*pred_m140_v))


#### Simple model

Write a ramdon simple model and see its performance. 

In [20]:
params0 = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.5,
    'eval_metric': 'tweedie-nloglik@1.5',
    'eta': 0.05, 
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'lambda': 1.0,
    'alpha': 1.0,
    'min_child_weight': 1,
}
# Train the model 

# evals = [(D_t, 'train'), (D_v, 'eval')] 
# !! early stopping is a grey area, many people use it on holdout, but it's soft cheating,
# !! ideally, try to avoid using it on final holdout, good to use on validation if you have extra untouched holdout

model0 = xgb.train(params = params0, 
                   dtrain = D_t, 
                   num_boost_round = 100, 
                   # early_stopping_rounds= 10, # ideally use it in V and not on final holdout
                   # evals=evals,  # evaluation datasets, the 'eval' dataset used for early stopping and quick view on perf improve 
                   # verbose_eval = 10 # display perf every xx rounds 
                  ) 

y_t_pred0 = model0.predict(D_t)

# Make predictions. And here you are actually predict the "losses $amount given ee".
y_v_pred0 = model0.predict(D_v,
                            # iteration_range=(0,mdl.best_iteration) # if you are using early stopping. You may want to score the model on the best iteration.
                            )

In [21]:
km_t0 = KeyMetrics(train_select['ncat_infl_adj']/ee_t, # actual pp
                   y_t_pred0/ ee_t, # predicted pp
                   ee_t)

# initiate KeyMetrics for this validation set:
km_v0 = KeyMetrics(val_select['ncat_infl_adj']/ee_v, # actual pp
                   y_v_pred0/ ee_v, # predicted pp
                   ee_v)
km_t0.gini(), km_v0.gini() 



(0.5841081634740499, 0.34208858377907925)

The model is overfitting, need to fine tuning.

#### Optuna tuning

Only use Gini score as our metric in this project

In [22]:
def get_gini(preds, dtrain):
    """
    customized metric of gini to be used within xgb.cv and optuna framework, need pumpkin package
    input: xgb.cv requires preds and dtrain as inputs, treat them as prediction and dmatrix, 
    output: a tuple of name and value, here, ('gini', gini_value)
    note, for gini in pumpkin, also need 'ee' as that gini computation need pure premium (loss or pred divided by ee)
          in our dmatrix before, 'ee' was saved as weight
    """
    actuals = dtrain.get_label()
    ee = dtrain.get_weight()
    km = KeyMetrics(actuals/ ee, # actual pp
                    preds/ ee, # predicted pp
                    ee)
    return 'gini', km.gini()   

def objective(trial):
    tweedie_variance_power = round(trial.suggest_float('tweedie_variance_power', 1.1, 1.8), 2)
    params = {
        'objective': 'reg:tweedie',
        'eval_metric': f'tweedie-nloglik@{tweedie_variance_power}',
        'eta': trial.suggest_float('eta', 0.003, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 6),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'lambda': trial.suggest_float('lambda', 1e-3, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'tweedie_variance_power': tweedie_variance_power}
    """ 
    cv_results = xgb.cv(params, 
                        dtrain = D_t, 
                        num_boost_round=1000, 
                        nfold=3, 
                        # folds = folds,
                        early_stopping_rounds=20,
                        metrics=[f'tweedie-nloglik@{tweedie_variance_power}'],
                        custom_metric=get_gini,
                        )
    nloglik = cv_results[f'test-tweedie-nloglik@{tweedie_variance_power}-mean'].iloc[-1]
    gini = cv_results[f'test-gini-mean'].iloc[-1]
    return nloglik, gini
     """
    # metric: gini only
    cv_results = xgb.cv(
        params,
        dtrain = D_t, 
        num_boost_round=500,
        nfold=3, 
        early_stopping_rounds=15,
        custom_metric=get_gini,
        verbose_eval=False
    )

    # --- REVISED METRIC ---
    best_iteration = cv_results.shape[0] - 1
    gini = cv_results['test-gini-mean'].iloc[best_iteration]
    
    # Store nloglik just to see it
    nloglik = cv_results[f'test-tweedie-nloglik@{tweedie_variance_power}-mean'].iloc[best_iteration]
    trial.set_user_attr('nloglik', nloglik)

    # Return the value to be *maximized* (by minimizing its negative)
    return -gini


def every_n_trials_callback(n):
    def callback(study, trial):
        if trial.number % n == 0:
            # print(f"Trial {trial.number}: Value={trial.value:.4f}, Params={trial.params}")
            print(
                f"Trial {trial.number}: "
                # f"Value={trial.value:.4f}, "
                # f"Best Value So Far={study.best_value:.4f}, "
                f"Params={trial.params}"
            )
    return callback

In [23]:
sampler = GPSampler(seed=42)
study = optuna.create_study(direction='minimize', sampler=sampler) # This is now correct
study.optimize(objective, n_trials=100, callbacks=[every_n_trials_callback(10)])

print(f"Best Gini: {-study.best_value}")

/tmp/ipykernel_11940/2101145732.py:1: ExperimentalWarning: GPSampler is experimental (supported from v3.6.0). The interface can change in the future.
  sampler = GPSampler(seed=42)
[I 2025-11-19 18:01:39,393] A new study created in memory with name: no-name-29eebb06-62b3-4d90-86e7-e81d780b3028
[I 2025-11-19 18:02:41,959] Trial 0 finished with value: -0.28326433333333334 and parameters: {'tweedie_variance_power': 1.3621780831931538, 'eta': 0.08412863932117019, 'max_depth': 5, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'lambda': 0.004207053950287938, 'alpha': 0.0017073967431528124, 'min_child_weight': 9}. Best is trial 0 with value: -0.28326433333333334.


Trial 0: Params={'tweedie_variance_power': 1.3621780831931538, 'eta': 0.08412863932117019, 'max_depth': 5, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'lambda': 0.004207053950287938, 'alpha': 0.0017073967431528124, 'min_child_weight': 9}


[I 2025-11-19 18:03:48,628] Trial 1 finished with value: -0.3442426666666667 and parameters: {'tweedie_variance_power': 1.5207805082202461, 'eta': 0.03592774284371348, 'max_depth': 3, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'lambda': 0.0070689749506246055, 'alpha': 0.005337032762603957, 'min_child_weight': 2}. Best is trial 1 with value: -0.3442426666666667.
[I 2025-11-19 18:04:25,946] Trial 2 finished with value: -0.36286466666666667 and parameters: {'tweedie_variance_power': 1.3129695700716764, 'eta': 0.018891292434997677, 'max_depth': 4, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578889518, 'lambda': 0.003613894271216527, 'alpha': 0.01474275315991467, 'min_child_weight': 4}. Best is trial 2 with value: -0.36286466666666667.
[I 2025-11-19 18:04:54,967] Trial 3 finished with value: -0.35357733333333335 and parameters: {'tweedie_variance_power': 1.4192489889519253, 'eta': 0.04708136413849345, 'max_depth': 3, 'subsample': 0.80569377536

Trial 10: Params={'tweedie_variance_power': 1.1, 'eta': 0.008484105917586703, 'max_depth': 3, 'subsample': 1.0, 'colsample_bytree': 0.7712101351638564, 'lambda': 3.827471461502173, 'alpha': 0.10673716175550578, 'min_child_weight': 5}


[I 2025-11-19 18:05:52,737] Trial 11 finished with value: -0.381437 and parameters: {'tweedie_variance_power': 1.1, 'eta': 0.012825041586842667, 'max_depth': 5, 'subsample': 1.0, 'colsample_bytree': 0.7480280070393053, 'lambda': 0.06474910393394935, 'alpha': 5.0373457050875325, 'min_child_weight': 7}. Best is trial 8 with value: -0.38184599999999996.
[I 2025-11-19 18:05:56,423] Trial 12 finished with value: -0.38156966666666664 and parameters: {'tweedie_variance_power': 1.6281401433310603, 'eta': 0.0077656246567877705, 'max_depth': 3, 'subsample': 0.9673684646577463, 'colsample_bytree': 0.6, 'lambda': 0.36771069160352465, 'alpha': 0.4693276958889041, 'min_child_weight': 8}. Best is trial 8 with value: -0.38184599999999996.
[I 2025-11-19 18:06:00,084] Trial 13 finished with value: -0.381407 and parameters: {'tweedie_variance_power': 1.1, 'eta': 0.009459981079155121, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 0.6, 'lambda': 10.0, 'alpha': 0.7478646786699775, 'min_child_weight'

...

Trial 80: Params={'tweedie_variance_power': 1.1, 'eta': 0.005845903032587589, 'max_depth': 5, 'subsample': 1.0, 'colsample_bytree': 1.0, 'lambda': 0.0010000000000000002, 'alpha': 0.2880148694215224, 'min_child_weight': 1}


[I 2025-11-19 18:12:23,699] Trial 81 finished with value: -0.38141933333333333 and parameters: {'tweedie_variance_power': 1.632753083668562, 'eta': 0.004359946813839516, 'max_depth': 3, 'subsample': 0.6, 'colsample_bytree': 1.0, 'lambda': 10.0, 'alpha': 0.0010000000000000002, 'min_child_weight': 10}. Best is trial 8 with value: -0.38184599999999996.
[I 2025-11-19 18:12:27,171] Trial 82 finished with value: -0.3814846666666667 and parameters: {'tweedie_variance_power': 1.5416228322977366, 'eta': 0.003, 'max_depth': 3, 'subsample': 1.0, 'colsample_bytree': 0.6, 'lambda': 10.0, 'alpha': 0.0010000000000000002, 'min_child_weight': 1}. Best is trial 8 with value: -0.38184599999999996.
[I 2025-11-19 18:12:31,356] Trial 83 finished with value: -0.381397 and parameters: {'tweedie_variance_power': 1.1, 'eta': 0.003, 'max_depth': 5, 'subsample': 0.6, 'colsample_bytree': 1.0, 'lambda': 10.0, 'alpha': 0.09170879716710659, 'min_child_weight': 10}. Best is trial 8 with value: -0.38184599999999996.
[I

Trial 90: Params={'tweedie_variance_power': 1.8, 'eta': 0.00519899232824848, 'max_depth': 4, 'subsample': 0.6, 'colsample_bytree': 1.0, 'lambda': 10.0, 'alpha': 9.999999999999993, 'min_child_weight': 1}


[I 2025-11-19 18:14:21,050] Trial 91 finished with value: -0.3813613333333333 and parameters: {'tweedie_variance_power': 1.3914888089118536, 'eta': 0.004707972509573401, 'max_depth': 4, 'subsample': 0.6, 'colsample_bytree': 1.0, 'lambda': 0.0010000000000000002, 'alpha': 10.0, 'min_child_weight': 1}. Best is trial 8 with value: -0.38184599999999996.
[I 2025-11-19 18:14:25,383] Trial 92 finished with value: -0.3810643333333334 and parameters: {'tweedie_variance_power': 1.6942758887301475, 'eta': 0.0053851782597521105, 'max_depth': 5, 'subsample': 0.6, 'colsample_bytree': 1.0, 'lambda': 10.0, 'alpha': 9.999999999999993, 'min_child_weight': 10}. Best is trial 8 with value: -0.38184599999999996.
[I 2025-11-19 18:14:28,920] Trial 93 finished with value: -0.3815476666666666 and parameters: {'tweedie_variance_power': 1.687969634715036, 'eta': 0.006286728296214488, 'max_depth': 3, 'subsample': 1.0, 'colsample_bytree': 1.0, 'lambda': 10.0, 'alpha': 10.0, 'min_child_weight': 1}. Best is trial 8 w

Best Gini: 0.38184599999999996


Save the best model and show its Gini score.

In [24]:
pareto_trials = study.best_trials

for trial in pareto_trials:
    print(f"Objectives: {trial.values}, Params: {trial.params}")

# refit final model w/ selected params & full training data
best_params=pareto_trials[0].params.copy()

best_params.update({'objective': 'reg:tweedie', 
                    'eval_metric': f'tweedie-nloglik@{round(best_params["tweedie_variance_power"], 2)}',
                    'random_state': 42})

model1 = xgb.train(params = best_params, 
                   dtrain = D_t, 
                   num_boost_round=100,
                   # early_stopping_rounds= 10, # ideally use it in V and not on final holdout
                   # evals=evals,  # evaluation datasets, the 'eval' dataset used for early stopping and quick view on perf improve 
                   # verbose_eval = 10 # display perf every xx rounds 
                  ) 


# Save the model
model1.save_model(f'./model/model5.1_cape_xgboost_tweedie_optuna_GP_2obj_0820_seed42_{version}.json')

Objectives: [-0.38184599999999996], Params: {'tweedie_variance_power': 1.2966541567811667, 'eta': 0.020117850889942897, 'max_depth': 3, 'subsample': 0.9208787923016158, 'colsample_bytree': 0.6298202574719083, 'lambda': 8.862326508576253, 'alpha': 1.2273800987852967, 'min_child_weight': 2}


In [25]:
# Make predictions
y_t_pred1 = model1.predict(D_t)
y_v_pred1 = model1.predict(D_v)

# initiate KeyMetrics for t set:
km_t1 = KeyMetrics(train_select['ncat_infl_adj']/ee_t, # actual pp
                   y_t_pred1/ ee_t, # predicted pp
                   ee_t)

# initiate KeyMetrics for this validation set:
km_v1 = KeyMetrics(val_select['ncat_infl_adj']/ee_v, # actual pp
                   y_v_pred1/ ee_v, # predicted pp
                   ee_v)

km_t1.gini(), km_v1.gini() 

(0.44089543543764576, 0.46562533455106936)

#### Holdout samples

In [26]:
X_test = test_select[ ls_predictors]
y_test = test_select[ 'ncat_infl_adj']
ee_test = test_select[ 'ee']
base_test = test_select['p0_alxc_wo_cape_pred']

D_test = xgb.DMatrix(X_test, label=y_test, weight=ee_test, enable_categorical=True) 
D_test.set_base_margin(np.log(ee_test*base_test))

Baseline Gini score

In [27]:
# initiate KeyMetrics for this test set:
km_base = KeyMetrics(test_select['ncat_infl_adj']/ee_test, # actual pp
                   base_test, # predicted pp
                   ee_test)
km_base.gini()

0.2730884360923187

Model prediction Gini score

In [28]:
y_test_pred1 = model1.predict(D_test)

# initiate KeyMetrics for t set:
# km_t1 = KeyMetrics(model_data.loc[model_data.tv == 'T', 'p0_alxc_amt_adj']/ ee_t, # actual pp
#                 y_t_pred1/ee_t, # predicted pp
#                 ee_t)

# initiate KeyMetrics for this validation set:
km_test = KeyMetrics(y_test/ee_test, # actual pp
                y_test_pred1/ee_test, # predicted pp
                ee_test)
print(km_test.gini())

0.28238978514963164


Save the results for holdout samples, used for future plotting.

In [29]:
pred_df = pd.DataFrame(
    {"y_test_pred1": y_test_pred1},
    index=X_test.index  # keep row alignment
)
# pred_df.to_csv(f"./output/{version}_y_test_pred1.csv", float_format="%.10g")
# test_select.to_csv(f"./output/{version}_test_select.csv")

#### Importance

In [30]:

def get_importance(xgb_model, print_flag = True, n = 20):
    """
    get xgboost model feature importance, a wrapper of original importance output
    input: trained xgb model object
    output: data frame of feature importance, sorted, display as percent
    """
    importance_gain = xgb_model.get_score(importance_type='gain') # a dict, importance_type can be 'gain'|'total_gain'|'weight'|'cover'|'total cover'. 
    # Convert to DataFrame for easier comparison and sorting
    importance_df = pd.DataFrame({
        'Feature': importance_gain.keys(),
        'Gain': importance_gain.values()
    })
    # Sort DataFrame by 'Gain'
    importance_df_sorted = importance_df.sort_values(by='Gain', ascending=False)

    # Calculate percentage importance
    importance_df_sorted['ImportancePct'] = round((importance_df_sorted['Gain'] / importance_df_sorted['Gain'].sum()) * 100,2)
    
    # clear index and round
    importance_df_sorted = importance_df_sorted.reset_index()
    
    if print_flag:
        print(importance_df_sorted[['Feature','ImportancePct']].head(n))
    else:
        return importance_df_sorted[['Feature','ImportancePct']]

In [31]:
get_importance(model1)

                                             Feature  ImportancePct
0                                  cape_roof_ponding           5.25
1                       cape_roof_ponding_confidence           4.56
2                    cape_primary_structure_latitude           4.45
3              cape_roof_condition_rating_confidence           4.27
4                 cape_accessory_structure_footprint           4.24
5                       cape_roof_sealing_confidence           4.13
6                     cape_accessory_structure_count           4.11
7                   cape_primary_structure_longitude           4.09
8                     cape_roof_streaking_confidence           3.97
9     cape_accessory_structure_roof_condition_rating           3.95
10                         cape_roof_tarp_confidence           3.74
11                                 cape_roof_sealing           3.65
12                                cape_roof_patching           3.64
13         cape_roof_material_degradation_confid